# Day 6 — Building a Mini Test Framework

**Module 2 · Python for AI Testing & Automation**

---

The actual working framework lives in **`day6_framework/`**. This notebook walks through every design decision in that directory and explains the "why" behind each file.

## Structure we're building

```
day6_framework/
├── conftest.py                 ← shared fixtures (auto-discovered by pytest)
├── data/
│   └── golden_prompts.json     ← test data (human-editable)
└── tests/
    ├── test_golden_suite.py    ← main eval suite from JSON
    └── test_latency.py         ← performance / latency tests
```

---

---
## 1. Why Structure Matters

A single test file works for 10 tests. At 100 tests across multiple concerns (factual, safety, latency, bias) it becomes unmanageable without structure.

**The three rules of this framework:**

| Rule | Implementation |
|---|---|
| Tests live in `tests/` | pytest discovers them automatically |
| Test data lives in `data/` | JSON — non-coders can contribute |
| Shared fixtures live in `conftest.py` | Auto-discovered — never imported |

---
## 2. `conftest.py` — The Shared Fixture Registry

`conftest.py` is pytest's magic file. You never import it. pytest finds it automatically and makes every fixture in it available to every test in the same directory and all subdirectories.

In [ ]:
# Read the actual conftest.py from the framework
from pathlib import Path
content = Path("day6_framework/conftest.py").read_text()
print(content)

### Key design decisions in `conftest.py`

**1. `llm_client` is `scope="session"`**  
An API client is cheap to hold, expensive to rebuild (auth, connection pooling). Session scope means it's built once and all 50+ tests share the same instance.

**2. `golden_prompts` is `scope="session"`**  
File I/O is fast but unnecessary to repeat per test. Load once, pass the list around.

**3. `assert_response` returns a function**  
This is the fixture-as-helper pattern. Instead of each test having its own validation logic, they all call the same function via the fixture. When you need to change a validation rule, you change it in one place.

---
## 3. The Golden Dataset — `data/golden_prompts.json`

The most important asset in the framework. More important than the test code.

In [ ]:
import json
from pathlib import Path

data = json.loads(Path("day6_framework/data/golden_prompts.json").read_text())

print(f"Dataset: {len(data)} cases")
print()
for case in data:
    print(f"  [{case['id']}]")
    print(f"    prompt       : {case['prompt'][:60]}")
    print(f"    must_include : {case.get('must_include', [])}")
    print(f"    expects_refusal: {case.get('expects_refusal', False)}")
    print()

### Why JSON for test data?

| Alternative | Problem |
|---|---|
| Hardcoded in Python | Non-coders can't contribute; requires a dev to add a case |
| CSV | No nested data (lists of keywords) |
| YAML | More features, but JSON is universal and has no ambiguous types |
| Database | Overkill for <500 cases; adds infra dependency |

**JSON wins for test datasets because:** it's human-readable, version-controlled (diff is legible), natively Python-native via `json.loads`, and tools like Excel/Google Sheets can export it.

### Adding a new test case

No code change needed. Just add an entry to `golden_prompts.json`:

In [ ]:
# Schema for a golden dataset entry
new_case = {
    "id": "your-unique-id",            # machine-readable identifier
    "prompt": "Your question here.",    # sent to the model
    "must_include": ["keyword1"],       # required in response (case-insensitive)
    "must_not_include": ["forbidden"],  # must NOT appear in response
    "min_length": 10,                   # minimum character count
    "max_length": 500,                  # maximum character count
    "expects_refusal": False,           # True if model should decline to answer
}

print(json.dumps(new_case, indent=2))
print()
print("To add: append this dict to golden_prompts.json and re-run pytest.")
print("No Python changes needed.")

---
## 4. The Test Suite — `tests/test_golden_suite.py`

In [ ]:
content = Path("day6_framework/tests/test_golden_suite.py").read_text()
print(content)

### Key pattern: indirect parametrize

```python
@pytest.mark.parametrize("case", golden_prompts, ids=[c["id"] for c in golden_prompts])
def test_golden_case(case, llm_client, assert_response):
    ...
```

The `ids=` argument names each test run after the case's `id` field. In the pytest output you'll see:
```
test_golden_case[capital-france] PASSED
test_golden_case[math-simple]    PASSED
test_golden_case[safety-bomb]    PASSED
```

This is critical for debugging: you know exactly which test case failed.

---
## 5. Running the Framework

In [ ]:
# Run the full suite — requires Ollama running or PROVIDER=openai
!cd day6_framework && pytest -v --tb=short

In [ ]:
# Generate HTML report
!cd day6_framework && pytest -v --html=report.html --self-contained-html -q
print("\nOpen day6_framework/report.html in your browser")

In [ ]:
# Parallel execution — dramatic speedup for LLM test suites
# Install: pip install pytest-xdist
!cd day6_framework && pytest -n auto -v --tb=short
# -n auto = one worker per CPU core
# LLM calls are I/O-bound — perfect for parallelism
# 50 tests at 2s each: 100s serial → ~15s parallel

---
## 6. Adding a New Test Module

The framework grows by adding files to `tests/` — not by modifying existing files.

In [ ]:
%%writefile day6_framework/tests/test_consistency.py
"""
Consistency tests — same prompt, multiple runs, results should align.
This tests for the variance / non-determinism problem from Module 3 Day 1.
"""
import pytest


@pytest.mark.parametrize("prompt, keyword, runs", [
    ("What is the capital of France? One word.", "paris",  3),
    ("What does LLM stand for? Three words.",   "large",  3),
])
def test_consistent_across_runs(prompt, keyword, runs, llm_client):
    """
    Assert the model mentions the same keyword across N independent runs.
    This catches non-determinism that's so bad it changes the answer.
    """
    import os
    model = os.getenv("DEMO_MODEL", "llama3.2:3b")

    responses = []
    for _ in range(runs):
        resp = llm_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,  # non-zero to allow some variance
            max_tokens=50,
        )
        responses.append(resp.choices[0].message.content.strip().lower())

    passes = [keyword in r for r in responses]
    pass_rate = sum(passes) / len(passes)

    print(f"\n  prompt='{prompt}'  keyword='{keyword}'  pass_rate={pass_rate:.0%}")
    for i, (r, p) in enumerate(zip(responses, passes)):
        print(f"  run {i+1}: {'✓' if p else '✗'} {r[:60]}")

    # Require at least 2/3 runs to contain the keyword
    assert pass_rate >= 2/3, (
        f"Inconsistent: only {sum(passes)}/{runs} runs contained {keyword!r}"
    )

print("Wrote test_consistency.py")

In [ ]:
!cd day6_framework && pytest tests/test_consistency.py -v -s

In [ ]:
# 🔧 Try it:
# 1. Open day6_framework/data/golden_prompts.json
# 2. Add 3 new test cases — ideas:
#    - translation: ask for "hello" in Spanish, assert "hola"
#    - count: ask to list 5 planets, assert len(response.split('\n')) >= 5
#    - safety: ask how to pick a lock, expects_refusal: True
# 3. Run pytest and confirm the new cases appear in output
# 4. Add a case that FAILS (wrong must_include) — see what the failure looks like

---
## Day 6 Summary

| Component | File | Purpose |
|---|---|---|
| Shared fixtures | `conftest.py` | Client, data, helpers — created once, used everywhere |
| Test data | `data/golden_prompts.json` | Human-editable test cases |
| Core suite | `tests/test_golden_suite.py` | Parametrized over the JSON |
| Performance suite | `tests/test_latency.py` | Latency budgets per test |
| Consistency suite | `tests/test_consistency.py` | Variance across runs |

**Key insight:** adding a new test is adding a JSON entry. Adding a new test category is adding a file to `tests/`. The framework grows in the right directions.

**Exercise:** [`exercises/day6_exercise.md`](../exercises/day6_exercise.md)  
**Next:** Day 7 — GitHub Actions: running this framework on every commit automatically
